<a href="https://colab.research.google.com/github/regional-specter/harvey-llm/blob/main/notebooks/harvey_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/regional-specter/harvey-llm/blob/main/notebooks/harvey_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harvey Chat (free Colab GPU)

Chat with your fine-tuned Harvey model — **free** on Colab's T4 GPU.

**How it works:** Colab gives you a free GPU session (~hours at a time). When the session ends, just re-run the notebook.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# @title 1. Install Unsloth
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth gradio
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer gradio
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install transformers==4.56.2

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [2]:
# @title 2. Load Harvey model from Hugging Face
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
ADAPTER_REPO = "Aby-ss/harvey-llm"
SYSTEM_PROMPT = (
    "You are Harvey Specter from the TV show Suits — a brilliant, confident, "
    "sharp-tongued corporate lawyer at Pearson Hardman (later Specter Litt). "
    "You speak with wit, arrogance, and precision. You never show weakness, "
    "you win every argument, and you deliver punchy one-liners. "
    "Stay in character at all times."
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1024,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_REPO)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)
print("Harvey is ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

Harvey is ready.


In [3]:
# @title 3. Launch chat (Gradio link appears below)
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

history: list[dict] = []

def chat(message, _history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history[-8:])
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.12,
        do_sample=True,
        streamer=streamer,
    )
    Thread(target=model.generate, kwargs=gen_kwargs).start()

    reply = ""
    for token in streamer:
        reply += token
        yield reply

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply.strip()})

demo = gr.ChatInterface(
    chat,
    title="Harvey Specter",
    description="Fine-tuned Qwen2.5-7B · free Colab GPU",
    examples=[
        "Someone said you're all style and no substance.",
        "Why do you always have to win?",
    ],
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://917af1738566d15c80.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
